# LogiTrack — Validação de Dados (Python + BigQuery + Great Expectations)

Notebook de conexão e preparação. Nele vamos: (1) conectar o Python ao BigQuery, (2) testar a conexão com uma consulta, (3) carregar as 5 tabelas em DataFrames do Pandas e (4) deixar o Great Expectations pronto para as validações do Desafio 3.

## 1. Instalação dos pacotes

Rode esta célula **uma vez**. Depois de instalar, **reinicie o kernel** (Kernel → Restart) e siga para a próxima célula.

In [1]:
# Instala os pacotes necessários (protobuf<7 evita o conflito que apareceu nas aulas)
!pip install -q -U google-cloud-bigquery db-dtypes pandas matplotlib seaborn great_expectations "protobuf<7"

^C


  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.


## 2. Imports

In [2]:
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

warnings.filterwarnings('ignore')
print('Imports OK')

Imports OK


## 3. Configuração e conexão com o BigQuery

**Como autenticar (escolha uma opção):**

- **Opção A — arquivo de credenciais (como nas aulas):** no Google Cloud, vá em *IAM e administrador → Contas de serviço*, crie uma conta com os papéis **Usuário de job do BigQuery** e **Leitor de dados do BigQuery**, gere uma chave JSON e coloque o caminho do arquivo em `CREDENTIALS_PATH` abaixo.
- **Opção B — login direto:** deixe `CREDENTIALS_PATH = ""` e, no terminal, rode `gcloud auth application-default login` (precisa do Google Cloud CLI instalado).

In [3]:
# ===== CONFIGURAÇÃO =====
PROJECT_ID = "addrafa"   # projeto do BigQuery
DATASET = "Prova"        # dataset onde estão as 5 tabelas

# Opção A: caminho do JSON da conta de serviço. Ex.: r"C:\Users\Rafael\Downloads\credencial.json"
# Opção B: deixe vazio ("") para usar o login do gcloud.
CREDENTIALS_PATH = ""

if CREDENTIALS_PATH:
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = CREDENTIALS_PATH

In [6]:
import pydata_google_auth

# Abre o navegador para você entrar com a conta Google que tem acesso ao projeto
credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"],
)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
print(f"Cliente criado para o projeto: {client.project}")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=262006177488-3425ks60hkk80fssi9vpohv88g6q1iqd.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fbigquery&state=at5sIhGvXti95hnRC2wdykb3sRnJN2&code_challenge=mtg2LipCPY2AkzIKVJPIPnHptiBdSbNrmS7dDMecwko&code_challenge_method=S256&prompt=consent&access_type=offline
Cliente criado para o projeto: addrafa


## 4. Teste de conexão

Consulta simples: se retornar **15000**, a conexão com a tabela fato está funcionando.

In [7]:
query_teste = f"""
    SELECT COUNT(*) AS total_linhas
    FROM `{PROJECT_ID}.{DATASET}.fato_entregas`
"""
df_teste = client.query(query_teste).to_dataframe(create_bqstorage_client=False)
print(df_teste)
print("Conectado ao BigQuery")

   total_linhas
0         15000
Conectado ao BigQuery


## 5. Carregando as 5 tabelas em DataFrames

In [8]:
def carregar_tabela(nome):
    """Lê uma tabela inteira do BigQuery e devolve um DataFrame do Pandas."""
    sql = f"SELECT * FROM `{PROJECT_ID}.{DATASET}.{nome}`"
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

df_fato           = carregar_tabela("fato_entregas")
df_cliente        = carregar_tabela("dim_cliente")
df_transportadora = carregar_tabela("dim_transportadora")
df_regiao         = carregar_tabela("dim_regiao")
df_data           = carregar_tabela("dim_data")

for nome, df in {"fato_entregas": df_fato, "dim_cliente": df_cliente,
                 "dim_transportadora": df_transportadora, "dim_regiao": df_regiao,
                 "dim_data": df_data}.items():
    print(f"{nome:20s} -> {df.shape[0]:>6} linhas x {df.shape[1]} colunas")

fato_entregas        ->  15000 linhas x 10 colunas
dim_cliente          ->    800 linhas x 4 colunas
dim_transportadora   ->      5 linhas x 3 colunas
dim_regiao           ->     10 linhas x 4 colunas
dim_data             ->    181 linhas x 6 colunas


In [12]:
# Conferência rápida da tabela fato
df_fato.head()

ImportError: cannot import name 'format_name' from 'pandas.io.formats.format' (c:\Users\Baian\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\io\formats\format.py)

   id_entrega  id_cliente  id_transportadora  id_regiao   id_data  \
0        1848          29                  1          1  20260422   
1        2256          81                  2          1  20260612   
2        6350         531                  5          1  20260101   
3        7153         600                  2          1  20260614   
4       12122         195                  1          1  20260430   

   prazo_prometido_dias  prazo_real_dias  valor_frete  custo_operacional  \
0                     3             <NA>        74.24              55.83   
1                     2             <NA>        63.99              51.41   
2                     7             <NA>        97.12              68.67   
3                     5             <NA>        29.21              21.90   
4                     4             <NA>        72.26              60.96   

  status_entrega  
0       Entregue  
1       Entregue  
2       Entregue  
3       Entregue  
4       Entregue  

In [10]:
# Tipos de dados e nulos da tabela fato
print(df_fato.dtypes)
print()
print(df_fato.isna().sum())

id_entrega                Int64
id_cliente                Int64
id_transportadora         Int64
id_regiao                 Int64
id_data                   Int64
prazo_prometido_dias      Int64
prazo_real_dias           Int64
valor_frete             float64
custo_operacional       float64
status_entrega           object
dtype: object

id_entrega               0
id_cliente               0
id_transportadora        0
id_regiao                0
id_data                  0
prazo_prometido_dias     0
prazo_real_dias         65
valor_frete             55
custo_operacional        0
status_entrega           0
dtype: int64


## 6. Great Expectations — verificação da instalação

In [11]:
import great_expectations as gx
print("Great Expectations versão:", gx.__version__)

Great Expectations versão: 1.24.1


---
Próximo passo: **Desafio 3** — as 4 validações (unicidade de `id_entrega`, nulos em `prazo_real_dias`, `custo_operacional > 0` e `prazo_prometido_dias > 0`) e o relatório HTML serão adicionados abaixo.